# CORAL : classification ordinale avec PyTorch

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moncoachdata/cercle-des-experts/blob/main/ml-classique-et-deep-learning/coral-classification-ordinale-pytorch/coral-classification-ordinale-pytorch.ipynb)

📖 **Le mini-cours complet** : [CORAL : comment forcer tes réseaux de neurones à respecter l'ordre naturel des classes](https://www.monshotdata.com/p/coral-classification-ordinale-pytorch)

🧭 **Le cours** : [ML classique et Deep Learning › 2.2 Régression et classification](https://www.monshotdata.com/mini-cours/ml-classique-et-deep-learning#h2sc22)

---

Prédire une note de 1 à 5 étoiles avec un classifieur classique, c'est traiter « 1 étoile » et « 5 étoiles » comme deux classes sans lien, alors que se tromper de 4 crans n'est pas du tout la même erreur que de se tromper d'un cran. Avec **CORAL**, ton réseau de neurones apprend que les classes ont un **ordre**, en changeant seulement les labels, la fonction d'activation et la loss. Tu vas le tester sur 7 105 avis d'hôtels.

**À la fin de ce notebook, tu sauras :**

- transformer des notes de 1 à 5 en labels binaires qui encodent l'ordre ;
- remplacer softmax + cross-entropy par sigmoïde + BCE et retrouver la note à partir des sorties du réseau ;
- comparer honnêtement un modèle ordinal à un classifieur standard, avec l'accuracy mais aussi l'erreur absolue moyenne.


### Avant de lancer

Les données (avis d'hôtels de TripAdvisor et leurs embeddings) sont téléchargées automatiquement depuis le dépôt GitHub : environ 27 Mo. Pas de GPU nécessaire, l'entraînement dure moins de deux minutes. Les graines aléatoires sont fixées pour des résultats reproductibles.

**Correction par rapport à la version diffusée avec le mini-cours :** l'encodage des labels binaires est corrigé (voir l'encadré dans la section 2). Les chiffres affichés ici diffèrent donc de ceux du mini-cours.

🧪 *Testé le 20 novembre 2026 avec Python 3.13, PyTorch 2.x, pandas et scikit-learn dans leurs versions de Colab.*

## 1. Mise en place

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import pickle

from tqdm import tqdm
from sklearn.model_selection import train_test_split

In [2]:
# Télécharge les fichiers nécessaires depuis le dépôt GitHub s'ils ne sont pas déjà présents
import os, urllib.request

BASE = "https://raw.githubusercontent.com/moncoachdata/cercle-des-experts/main/ml-classique-et-deep-learning/coral-classification-ordinale-pytorch"
for f in ['review_data.pkl', 'embeddings.pkl']:
    if not os.path.exists(f):
        os.makedirs(os.path.dirname(f) or ".", exist_ok=True)
        urllib.request.urlretrieve(f"{BASE}/{f}", f)

## 2. Encoder les notes en labels binaires

Idée de CORAL : au lieu d'un label unique parmi 5 classes, on pose **4 questions binaires** ordonnées : « la note est-elle > 1 ? », « > 2 ? », « > 3 ? », « > 4 ? ». Une note de 3 étoiles donne donc `[1, 1, 0, 0]` : deux « oui » suivis de « non ». Les labels sont toujours une suite de 1 puis de 0, ce qui garantit la **cohérence des rangs**.

> **Correction :** la version d'origine écrivait `label` « oui » au lieu de `label - 1`. Avec 5 classes et 4 colonnes, les notes 4 et 5 recevaient le même encodage `[1, 1, 1, 1]` et la note 5 ne pouvait jamais être prédite. L'encodage correct a `note - 1` valeurs à 1.

In [3]:
def transform_labels_to_binary(y, num_classes):
    # note k (de 1 à num_classes)  ->  k-1 valeurs à 1 suivies de zéros
    transformed_labels = torch.zeros(y.shape[0], num_classes-1)

    for idx, label in enumerate(y):
        transformed_labels[idx, 0:label-1] = 1
    return transformed_labels

## 3. Les données

Un sous-ensemble du jeu de données *TripAdvisor Hotel Reviews* (Kaggle) : 7 105 avis, répartis de façon **équilibrée** entre les notes de 1 à 5. Le texte a déjà été transformé en embeddings de dimension 768 (avec un modèle SentenceTransformers de type DistilBERT) : le fichier `embeddings.pkl` est fourni, tu n'as pas besoin de recalculer quoi que ce soit.

In [4]:
df = pd.read_pickle("review_data.pkl")

In [5]:
with open("embeddings.pkl", "rb") as h:
    embeddings = pickle.load(h)
    
y = df.Rating.values

On sépare entraînement (80 %) et test (20 %), puis on convertit en tenseurs PyTorch. La graine fixe le découpage pour que les résultats soient reproductibles.

In [6]:
x_train, x_test, y_train, y_test = train_test_split(embeddings, y, test_size=0.2, random_state=42)

In [7]:
embeddings_train = torch.tensor(x_train)
embeddings_test = torch.tensor(x_test)

y_train = torch.tensor(y_train)
y_test = torch.tensor(y_test)

num_classes = len(y_test.unique())

transformed_labels = transform_labels_to_binary(y_train, num_classes)

## 4. Le modèle ordinal (CORAL)

Le réseau a **deux couches cachées** de 30 neurones et **4 sorties** (une par question binaire). Particularité CORAL : chaque sortie reçoit son **propre biais** indépendant, et on applique une **sigmoïde** (pas un softmax).

In [8]:
class OrdinalClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(OrdinalClassifier, self).__init__()
        
        # définition des couches
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, num_classes - 1)
        
        # biais indépendants
        self.biases = nn.Parameter(torch.zeros(num_classes - 1))

    def forward(self, x):
        
        # calcul de la sortie des couches
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        output = self.fc3(x)
        
        # ajout du terme de biais
        output += self.biases
        
        # retourne le sigmoid
        return torch.sigmoid(output)

On entraîne avec la **BCE** (Binary Cross-Entropy) et l'optimiseur Adam.

In [9]:
torch.manual_seed(42)

model = OrdinalClassifier(embeddings_train.shape[1], 30, num_classes)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [10]:
# Définition des paramètres d'entraînement
num_epochs = 10
batch_size = 64
num_batches = len(embeddings_train) // batch_size

### La boucle d'entraînement

À chaque époque, on parcourt les mini-batchs, puis on mesure l'accuracy sur le test. Pour retrouver la note à partir des 4 sorties : on seuille à 0,5 pour obtenir 4 labels binaires, on les **additionne** (le nombre de « oui » = note − 1), puis on ajoute 1.

In [11]:
# Boucle d'entraînement
for epoch in range(num_epochs):
    running_loss = 0.0
    
    # Itération par mini-batch
    for batch in range(num_batches):
        start_idx = batch * batch_size
        end_idx = min((batch + 1) * batch_size, len(embeddings_train))
        inputs = embeddings_train[start_idx:end_idx]
        
        # Génération des labels binaires pour le mini-batch
        binary_labels = transformed_labels[start_idx:end_idx]
        
        # Remise à zéro des gradients
        optimizer.zero_grad()
        
        # Passe avant (forward pass)
        outputs = model(inputs)
        
        # Calcul de la loss
        loss = criterion(outputs, binary_labels.float())
        
        # Passe arrière et optimisation (backward pass)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    # Précision
    test_outputs = model(embeddings_test)
    test_binary = (test_outputs>0.5).to(int)
    test_labels = torch.sum(test_binary, 1) + 1    # nombre de « oui » + 1 = note prédite
    accuracy = float(sum(test_labels == y_test)/y_test.shape[0])
    
    # Affichage de la loss moyenne pour chaque epoch
    print(f"Epoch {epoch+1}, Loss: {round(running_loss / num_batches, 3)}, Accuracy = {round(accuracy, 3)}")

Epoch 1, Loss: 0.434, Accuracy = 0.456


Epoch 2, Loss: 0.327, Accuracy = 0.488


Epoch 3, Loss: 0.311, Accuracy = 0.498


Epoch 4, Loss: 0.302, Accuracy = 0.5


Epoch 5, Loss: 0.295, Accuracy = 0.498


Epoch 6, Loss: 0.291, Accuracy = 0.508


Epoch 7, Loss: 0.287, Accuracy = 0.513


Epoch 8, Loss: 0.283, Accuracy = 0.514


Epoch 9, Loss: 0.28, Accuracy = 0.513


Epoch 10, Loss: 0.277, Accuracy = 0.511


In [12]:
ordinal_model = model   # on garde ce modèle pour la comparaison finale

## 5. Le modèle de référence : classification standard

Même architecture, mais 5 sorties, **softmax** (inclus dans `CrossEntropyLoss`) et cross-entropy. C'est la méthode habituelle, qui ignore l'ordre des classes. Note : le taux d'apprentissage est de 0,01 ici contre 0,001 pour le modèle ordinal, comme dans le mini-cours.

In [13]:
class NormalClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(NormalClassifier, self).__init__()
        
        # définition des couches
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        
        # calcul de la sortie des couches
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        output = self.fc3(x)
        return output

In [14]:
torch.manual_seed(42)

model = NormalClassifier(embeddings_train.shape[1], 30, num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [15]:
# Définition des paramètres d'entraînement
num_epochs = 10
batch_size = 64
num_batches = len(embeddings_train) // batch_size

In [16]:
# Boucle d'entraînement
for epoch in range(num_epochs):
    running_loss = 0.0
    
    # Itération par mini-batch
    for batch in range(num_batches):
        start_idx = batch * batch_size
        end_idx = min((batch + 1) * batch_size, len(embeddings_train))
        inputs = embeddings_train[start_idx:end_idx]
        
        # Génération des labels pour le mini-batch
        binary_labels = y_train[start_idx:end_idx] - 1
        
        # Remise à zéro des gradients
        optimizer.zero_grad()
        
        # Passe avant (forward pass)
        outputs = model(inputs)
        
        # Calcul de la loss
        loss = criterion(outputs, binary_labels)
        
        # Passe arrière et optimisation (backward pass)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    test_outputs = model(embeddings_test)
    accuracy = float(sum(torch.argmax(test_outputs, 1) + 1 == y_test)/y_test.shape[0])
    
    # Affichage de la loss moyenne pour chaque epoch
    print(f"Epoch {epoch+1}, Loss: {round(running_loss / num_batches, 3)}, Accuracy = {round(accuracy, 3)}")

Epoch 1, Loss: 1.185, Accuracy = 0.502


Epoch 2, Loss: 1.088, Accuracy = 0.484


Epoch 3, Loss: 1.056, Accuracy = 0.488


Epoch 4, Loss: 1.035, Accuracy = 0.49


Epoch 5, Loss: 1.016, Accuracy = 0.495


Epoch 6, Loss: 1.002, Accuracy = 0.488


Epoch 7, Loss: 0.985, Accuracy = 0.506


Epoch 8, Loss: 0.988, Accuracy = 0.501


Epoch 9, Loss: 0.978, Accuracy = 0.494


Epoch 10, Loss: 0.959, Accuracy = 0.5


In [17]:
normal_model = model

## 6. Comparer les deux modèles

L'accuracy ne dit pas tout : se tromper d'**un cran** (3 étoiles au lieu de 4) est bien moins grave que de se tromper de **quatre** (1 au lieu de 5). L'**erreur absolue moyenne** (MAE, en nombre d'étoiles) mesure justement cet écart : c'est la métrique naturelle d'un problème ordinal.

In [18]:
with torch.no_grad():
    pred_ordinal = (ordinal_model(embeddings_test) > 0.5).to(int).sum(1) + 1
    pred_normal = normal_model(embeddings_test).argmax(1) + 1

for nom, pred in [("Classification standard (cross-entropy)", pred_normal), ("CORAL (labels binaires + BCE)", pred_ordinal)]:
    accuracy = (pred == y_test).float().mean().item()
    mae = (pred - y_test).abs().float().mean().item()
    print(f"{nom:42s} accuracy = {accuracy:.3f}   MAE = {mae:.3f} étoile")

Classification standard (cross-entropy)    accuracy = 0.500   MAE = 0.652 étoile
CORAL (labels binaires + BCE)              accuracy = 0.511   MAE = 0.576 étoile


### Ce qu'il faut retenir

Sur ce découpage, CORAL fait un peu mieux en accuracy et **nettement mieux en erreur absolue moyenne** : même quand il se trompe, il se trompe moins loin de la bonne note. En répétant l'expérience sur cinq découpages différents, l'avantage se confirme (environ 0,52 contre 0,48 d'accuracy, et 0,56 contre 0,63 étoile de MAE). Essaie d'autres graines (`random_state`) pour le vérifier toi-même.

---

## Pour aller plus loin

- [Booste tes modèles ML sans changer d'algorithme (Partie 2)](https://www.monshotdata.com/p/techniques-ml-avancees-focal-loss-breathing-kmeans-features-cycliques) — la technique précédente de la série : la Focal Loss
- [Régression Logistique : Plongée dans la logique de la perte logarithmique](https://www.monshotdata.com/p/regression-logistique-perte-logarithmique) — la BCE, brique de base de CORAL
- [Calibration des modèles ML (Partie 1)](https://www.monshotdata.com/p/calibration-modeles-machine-learning) — vérifier que les probabilités du modèle sont fiables